## hybrid retriever -combining dense and sparse retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from typing import List

In [4]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

C:\Users\SamarjitMahi\AppData\Local\Temp\ipykernel_33736\1771132316.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
c:\Users\SamarjitMahi\OneDrive - ZapCom Solutions Pvt. ltd\Pictures\genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8556.07it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
-----------

In [ ]:
### Custom Ensemble Retriever using Reciprocal Rank Fusion (RRF)

class EnsembleRetriever(BaseRetriever):
    """Combines multiple retrievers using Reciprocal Rank Fusion."""
    def __init__(self, retrievers: List, weights: List[float] = None):
        super().__init__()
        self.retrievers = retrievers
        self.weights = weights or [1.0] * len(retrievers)

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        # Collect results from each retriever
        all_results = [r.invoke(query) for r in self.retrievers]

        # Reciprocal Rank Fusion scoring
        scores: dict = {}
        for docs, weight in zip(all_results, self.weights):
            for rank, doc in enumerate(docs):
                key = doc.page_content
                if key not in scores:
                    scores[key] = {"doc": doc, "score": 0.0}
                scores[key]["score"] += weight / (rank + 60)

        # Return sorted by score
        ranked = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
        return [item["doc"] for item in ranked]

In [ ]:
### Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3  # top-k documents to retrieve

## Step 4: Combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]  # 70% dense, 30% sparse
)

In [7]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000028BC66CB8C0>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000028BCA9517F0>, k=3)], weights=[0.5, 0.5])

In [8]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


### RAG Pipeline with hybrid retriever

In [16]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

# Step 6: LLM (using Groq instead of OpenAI)
llm = init_chat_model("groq:llama-3.1-8b-instant", temperature=0.2)

# Create RAG chain using LangChain 1.x pattern
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": hybrid_retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Step 7: Ask a question
query = "How can I build an app using LLMs?"
response = rag_chain.invoke(query)

# Step 8: Output
print("✅ Answer:\n", response)

# Get source documents separately for display
source_docs = hybrid_retriever.invoke(query)
print("\n📄 Source Documents:")
for i, doc in enumerate(source_docs):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 Based on the context provided, you can build an app using LLMs (Large Language Models) with the help of LangChain. LangChain is a platform that enables developers to build LLM applications, including agentic AI applications. 

Here's a general outline of how you can build an app using LLMs with LangChain:

1. **Choose a LangChain component**: LangChain has various components, including retrievers, which can be used to fetch relevant information from a database or the internet.
2. **Select a retriever**: LangChain has multiple types of retrievers, which can be used to fetch information from different sources. You can choose the one that best fits your use case.
3. **Integrate with a vector database**: If you need to perform semantic search, you can integrate LangChain with a vector database like Pinecone. Pinecone allows you to store and retrieve vectors, which can be used for semantic search.
4. **Develop your LLM application**: Once you have set up the necessary components,